In [5]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import re
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# 1) Load data
abstracts = pd.read_csv('abstracts.txt', delimiter='\t', header=None, names=['abstract'])
edgelist = pd.read_csv('edgelist.txt', delimiter=',', header=None, names=['source', 'target'])
test_edges = pd.read_csv('test.txt', delimiter=',', header=None, names=['source', 'target'])

# 2) Text preprocessing
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
def clean_text(text):
    text = re.sub(r'[^a-zA-Z]', ' ', text).lower()
    tokens = [w for w in text.split() if w not in stop_words]
    stems = [stemmer.stem(w) for w in tokens if w not in string.punctuation]
    return ' '.join(stems)

abstracts['cleaned'] = abstracts['abstract'].apply(clean_text)

# 3) Sentence-BERT embeddings
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = sbert_model.encode(
    abstracts['cleaned'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)
abstracts['vector'] = list(embeddings)

# 4) Smart negative sampling (as before)
src, tgt = edgelist['source'].values, edgelist['target'].values
forbidden = set(zip(src, tgt)) | set(zip(tgt, src))
n_nodes, num_pos = abstracts.shape[0], len(edgelist)
negatives = set()
while len(negatives) < num_pos:
    u, v = np.random.randint(n_nodes), np.random.randint(n_nodes)
    if u != v and (u, v) not in forbidden:
        negatives.add((u, v))
neg_src, neg_tgt = zip(*negatives)
negative_pairs = pd.DataFrame({'source': neg_src, 'target': neg_tgt, 'label': 0})

# 5) Prepare training DataFrame
positive_pairs = edgelist.copy(); positive_pairs['label'] = 1
training_data = pd.concat([positive_pairs, negative_pairs]).sample(frac=1).reset_index(drop=True)

# 6) Cosine-similarity feature
def cosine_similarity(a, b, eps=1e-10):
    return np.dot(a, b) / (max(np.linalg.norm(a)*np.linalg.norm(b), eps))

training_data['similarity'] = training_data.apply(
    lambda r: cosine_similarity(abstracts.loc[r.source, 'vector'],
                                abstracts.loc[r.target, 'vector'])
, axis=1)

# 7) Train & evaluate
X_train, X_val, y_train, y_val = train_test_split(
    training_data[['similarity']], training_data['label'], test_size=0.2, random_state=42
)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
val_preds = lr.predict_proba(X_val)[:,1]
print('Validation log loss:', log_loss(y_val, val_preds))

# 8) Predict on test set
test_edges['similarity'] = test_edges.apply(
    lambda r: cosine_similarity(
        abstracts.loc[r.source, 'vector'],
        abstracts.loc[r.target, 'vector']
    ), axis=1
)
preds = lr.predict_proba(test_edges[['similarity']])[:,1]
submission = pd.DataFrame({'ID': range(len(test_edges)), 'Label': preds})
submission.to_csv('submission.csv', index=False)
print("Saved submission.csv")


Validation log loss: 0.48931090182306647
Saved submission.csv
